# Prepare Clinical data

In [39]:
LONG_RESP_THRESHOLD = 12
DROP_BOR = True

In [40]:
import polars as pl
import numpy as np
import pandas as pd

source_map = {
    'doi:10.1016/j.cell.2015.07.061': 'Hugo et al.',
    'doi:10.1172/JCI78954DS1': 'Kwong et al.',
    'doi:10.1158/1078-0432.CCR-18-0720': 'Yan et al.',
    'doi:10.3390/cancers11081203': 'Louveau et al.',
    'doi:10.1038/ncomms6694': 'Long et al.',
    'doi:10.1158/1078-0432.CCR-13-3122': 'Rizos et al.',
    "doi:10.3390/cancers12082224": 'Blateau et al.',
    "doi:10.1200/PO.16.00054": 'Catalanotti et al.',
    'doi:10.1158/2159-8290.CD-13-0617': 'Van Allen at al.'
}

clinical = pl.read_csv(f"../dataset/original/clinical.csv")
clinical = clinical.with_columns(pl.col('source').replace(source_map))
clinical = clinical.drop(['id', 'creation_datetime', 'original_patientID', 'OS_status', 'OS_month', 'CNA_data', 'SNV_data', 'GEX_data'])
if DROP_BOR == True:
    clinical = clinical.drop(['BOR'])
clinical

patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_status,PFS_month,drug,BRAF_mut,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source
str,str,i64,str,str,str,str,f64,str,str,str,str,str,str
"""BS_000""","""male""",56,"""IV""",null,"""normal""","""1""",30.5,"""dabrafenib + trametinib""","""V600E""","""no""","""no""","""no""","""Blateau et al."""
"""BS_001""","""male""",86,"""IV""",null,"""normal""","""0""",24.1,"""dabrafenib""","""V600E""","""no""","""no""","""no""","""Blateau et al."""
"""BS_002""","""female""",47,"""IV""",null,"""normal""","""0""",14.1,"""dabrafenib + trametinib""","""V600E""","""no""","""no""","""no""","""Blateau et al."""
"""BS_003""","""female""",50,"""IV""",null,null,"""1""",1.6,"""vemurafenib""","""V600E""","""no""","""no""","""no""","""Blateau et al."""
"""BS_004""","""female""",47,"""IV""",null,"""elevated""","""1""",11.9,"""dabrafenib + trametinib""","""V600K""","""no""","""no""","""no""","""Blateau et al."""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HL_40""","""male""",47,"""IV""","""M1C""",null,"""1""",3.0,"""vemurafenib""","""V600E""","""no""","""no""","""no""","""Hugo et al."""
"""HL_41""","""male""",39,"""IV""","""M1A""",null,"""1""",4.0,"""vemurafenib""","""V600E""","""no""","""no""","""no""","""Hugo et al."""
"""HL_42""","""male""",84,"""IV""","""M1C""",null,"""1""",8.0,"""dabrafenib""","""V600E""","""no""","""no""","""no""","""Hugo et al."""


In [41]:
clinical.select(pl.col('source').unique())

source
str
"""Van Allen at al."""
"""Yan et al."""
"""Blateau et al."""
"""Louveau et al."""
"""Kwong et al."""
"""Long et al."""
"""Catalanotti et al."""
"""Hugo et al."""
"""Rizos et al."""


In [42]:
# def get_pfs_label(row):
#     months = row['PFS_month']
#     event = row['PFS_status']
    
#     if months >= LONG_RESP_THRESHOLD:
#         return 2  # Long responder
#     elif months < 6 and event == 1:
#         return 0   # Non responder
#     elif 6 <= months < LONG_RESP_THRESHOLD and event == 1:
#         return 1 # Intermediate
#     # else:
#     #     return np.nan   # Undetermind
#     # Censored cases - assign based on lower bound
#     elif months < 6 and event == 0:
#         return 0  # Conservative: assume non-responder (survived AT LEAST x months)
#     elif 6 <= months < LONG_RESP_THRESHOLD and event == 0:
#         return 1  # Conservative: assume intermediate
    
# clinical_gex['pfs_label'] = clinical_gex.apply(get_pfs_label, axis=1)

# # clinical_gex = clinical_gex.drop(['PFS_status', 'PFS_month'], axis=1)
# clinical_gex.iloc[:, :10]

## Add Target label

In [43]:
# Check unexpected PFS_status values
print(clinical['PFS_status'].value_counts())


shape: (3, 2)
┌────────────┬───────┐
│ PFS_status ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ 0          ┆ 74    │
│ 1          ┆ 341   │
│ <NA>       ┆ 2     │
└────────────┴───────┘


In [44]:
clinical = clinical.with_columns(
    pl.when(pl.col('PFS_month') >= LONG_RESP_THRESHOLD)
    .then(pl.lit(2))
    .when((pl.col('PFS_month') < 6) & (pl.col('PFS_status') == "1"))
    .then(pl.lit(0))
    .when((pl.col('PFS_month') >= 6) & (pl.col('PFS_month') < LONG_RESP_THRESHOLD) & (pl.col('PFS_status') == "1"))
    .then(pl.lit(1))
    .when((pl.col('PFS_month') < 6) & (pl.col('PFS_status') == "0"))
    .then(pl.lit(0))
    .when((pl.col('PFS_month') >= 6) & (pl.col('PFS_month') < LONG_RESP_THRESHOLD) & (pl.col('PFS_status') == "0"))
    .then(pl.lit(1))
    # .otherwise(None)
    .alias('pfs_label')
).drop(['PFS_status', 'PFS_month']).drop_nulls('pfs_label')
clinical

patientID,sex,age,AJCC_stage,M_stage,LDH,drug,BRAF_mut,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source,pfs_label
str,str,i64,str,str,str,str,str,str,str,str,str,i32
"""BS_000""","""male""",56,"""IV""",null,"""normal""","""dabrafenib + trametinib""","""V600E""","""no""","""no""","""no""","""Blateau et al.""",2
"""BS_001""","""male""",86,"""IV""",null,"""normal""","""dabrafenib""","""V600E""","""no""","""no""","""no""","""Blateau et al.""",2
"""BS_002""","""female""",47,"""IV""",null,"""normal""","""dabrafenib + trametinib""","""V600E""","""no""","""no""","""no""","""Blateau et al.""",2
"""BS_003""","""female""",50,"""IV""",null,null,"""vemurafenib""","""V600E""","""no""","""no""","""no""","""Blateau et al.""",0
"""BS_004""","""female""",47,"""IV""",null,"""elevated""","""dabrafenib + trametinib""","""V600K""","""no""","""no""","""no""","""Blateau et al.""",1
…,…,…,…,…,…,…,…,…,…,…,…,…
"""HL_40""","""male""",47,"""IV""","""M1C""",null,"""vemurafenib""","""V600E""","""no""","""no""","""no""","""Hugo et al.""",0
"""HL_41""","""male""",39,"""IV""","""M1A""",null,"""vemurafenib""","""V600E""","""no""","""no""","""no""","""Hugo et al.""",0
"""HL_42""","""male""",84,"""IV""","""M1C""",null,"""dabrafenib""","""V600E""","""no""","""no""","""no""","""Hugo et al.""",1


In [45]:
clinical['pfs_label'].value_counts(sort=True)

pfs_label,count
i32,u32
0,223
1,106
2,86


## OHE 'drug' and 'BRAF_mut'

In [46]:
# clinical = clinical.to_pandas()
# drug_dummies = clinical['drug'].str.get_dummies(sep='+')
# clinical = pd.concat([clinical.drop('drug', axis=1), drug_dummies], axis=1)
# braf_mut_dummies = clinical['BRAF_mut'].str.get_dummies(sep=';')
# clinical = pd.concat([clinical.drop('BRAF_mut', axis=1), braf_mut_dummies], axis=1)
# clinical
clinical = clinical.to_pandas()

drug_series = clinical['drug'].fillna('').str.replace(r'\s*\+\s*', '+', regex=True).str.strip()
drug_dummies = drug_series.str.get_dummies(sep='+').drop(columns=[''], errors='ignore')
clinical = pd.concat([clinical.drop('drug', axis=1), drug_dummies], axis=1)

braf_series = clinical['BRAF_mut'].fillna('').str.replace(r'\s*;\s*', ';', regex=True).str.strip()
braf_mut_dummies = braf_series.str.get_dummies(sep=';').drop(columns=[''], errors='ignore')
clinical = pd.concat([clinical.drop('BRAF_mut', axis=1), braf_mut_dummies], axis=1)
clinical = clinical.drop(columns=['nan'], errors='ignore')
clinical

,patientID,sex,age,AJCC_stage,M_stage,LDH,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source,...,dabrafenib,trametinib,vemurafenib,C195W,K601I,MND,R558Q,V600E,V600K,V600R
0,BS_000,male,56,IV,None,normal,no,no,no,Blateau et al.,...,1,1,0,0,0,0,0,1,0,0
1,BS_001,male,86,IV,None,normal,no,no,no,Blateau et al.,...,1,0,0,0,0,0,0,1,0,0
2,BS_002,female,47,IV,None,normal,no,no,no,Blateau et al.,...,1,1,0,0,0,0,0,1,0,0
3,BS_003,female,50,IV,None,None,no,no,no,Blateau et al.,...,0,0,1,0,0,0,0,1,0,0
4,BS_004,female,47,IV,None,elevated,no,no,no,Blateau et al.,...,1,1,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
410,HL_40,male,47,IV,M1C,None,no,no,no,Hugo et al.,...,0,0,1,0,0,0,0,1,0,0
411,HL_41,male,39,IV,M1A,None,no,no,no,Hugo et al.,...,0,0,1,0,0,0,0,1,0,0
412,HL_42,male,84,IV,M1C,None,no,no,no,Hugo et al.,...,1,0,0,0,0,0,0,1,0,0
413,HL_43,female,41,IV,M1C,None,no,no,no,Hugo et al.,...,0,0,1,0,0,0,0,1,0,0


## Save

In [47]:
clinical.to_csv(f"../dataset/created/clinical.csv")